<a href="https://colab.research.google.com/github/agargya123/airbnb-reviews-nlp/blob/transformer-colab-notebook/notebooks/03_Transformer_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook is run in Google Colab so that we can utilize GPU for processing. Go to Runtime -> Change Runtime Type -> T4 GPU -> Save

Clone your repository

In [1]:
# COLAB SETUP — run first
!git clone https://github.com/agargya123/airbnb-reviews-nlp.git

Cloning into 'airbnb-reviews-nlp'...
remote: Enumerating objects: 187, done.
remote: Counting objects: 100% (187/187), done.
remote: Compressing objects: 100% (128/128), done.
remote: Total 187 (delta 91), reused 141 (delta 51), pack-reused 0 (from 0)
Receiving objects: 100% (187/187), 7.24 MiB | 17.48 MiB/s, done.
Resolving deltas: 100% (91/91), done.
Filtering content: 100% (13/13), 694.59 MiB | 5.62 MiB/s, done.


Import data from 2 key boroughs

In [3]:
import os
os.chdir('/content/airbnb-reviews-nlp')

In [4]:
from pathlib import Path
import pandas as pd

# this finds the project root regardless of where the notebook is
ROOT = Path('/content/airbnb-reviews-nlp')
SAMPLES_DIR = ROOT / "data" / "samples"
#Load the datasets
manhattan_sample_df        = pd.read_csv(SAMPLES_DIR / "manhattan_sample.csv")
brooklyn_sample_df = pd.read_csv(SAMPLES_DIR / "brooklyn_sample.csv")

In [7]:
# Only needed if running in Colab
import sys
if 'google.colab' in sys.modules:
    !pip install transformers scipy

Check the installed versions, these can be added to the requirements.txt     [OPTIONAL]

In [ ]:
import transformers
import torch
import scipy

print(f"transformers=={transformers.__version__}")
print(f"torch=={torch.__version__}")
print(f"scipy=={scipy.__version__}")

### 1) Installs and Imports

In [8]:

import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from scipy.special import softmax
import time
import os

# 1. DEVICE SETUP

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
# Local: will print 'cpu'
# Colab with GPU: will print 'cuda'


Using device: cuda


### 2) Model Selection :

We select a model cardiffnlp/twitter-roberya-base-sentiment-latest that is able to provide us three categories of review classification - Positive, Negative, Neutral instead of the usual Positive/Negative.


**Why is this important?**

Airbnb has a well-documented "grade inflation" problem — guests are socially conditioned to leave positive reviews even when the experience was just okay, because they don't want to hurt a host's livelihood.

So a review might say:
`"The apartment was fine. It was clean and the location was decent. Check-in was a bit confusing but we figured it out."`

This is clearly not a positive experience — "fine", "decent", "a bit confusing" are all lukewarm signals. But it's also not negative. The guest isn't complaining, they're describing a mediocre experience.

Without a Neutral class (e.g. using distilbert-sst-2 binary):
The model is forced to pick and will almost certainly call this 'Positive' because there are no overtly negative words. Your host now thinks this listing is performing well.

With a Neutral class (cardiffnlp):
The model correctly labels this Neutral. This is valuable, and actionable information.

Customers are more comfortable reviewing negatively for products but apprehensive of doing so for people. For Airbnb, this third category captures the "quietly dissatisified" class.

In [9]:
# 2. LOAD MODEL

MODEL_NAME = "cardiffnlp/twitter-roberta-base-sentiment-latest"

print(f"Loading model: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
model = model.to(device)
model.eval()

# Labels: 0=Negative, 1=Neutral, 2=Positive
labels = ['Negative', 'Neutral', 'Positive']
print(f"Labels: {labels}")

Loading model: cardiffnlp/twitter-roberta-base-sentiment-latest


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/501M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.pooler.dense.bias       | UNEXPECTED |  | 
roberta.pooler.dense.weight     | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Labels: ['Negative', 'Neutral', 'Positive']


### 3) Defining Inference Functions

In [10]:
# 3.1 INFERENCE FUNCTION

def get_sentiment_batch(texts, batch_size=32):
    """
    Runs sentiment analysis on a list of texts in batches.
    Returns list of dicts with label + confidence scores.

    Args:
        texts       : list of review strings
        batch_size  : 32 works well on Colab GPU, reduce to 8-16 for local CPU

    Returns:
        List of dicts: [{label, score_negative, score_neutral, score_positive}]
    """
    results = []
    total_batches = len(texts) // batch_size + 1

    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i + batch_size]

        # Tokenize — truncate to 512 tokens (model max)
        encoded = tokenizer(
            batch_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512
        ).to(device)

        with torch.no_grad():
            output = model(**encoded)

        # Convert logits to probabilities
        scores = softmax(output.logits.cpu().numpy(), axis=1)

        for score in scores:
            results.append({
                'sentiment_label'    : labels[np.argmax(score)],
                'score_negative'     : round(float(score[0]), 4),
                'score_neutral'      : round(float(score[1]), 4),
                'score_positive'     : round(float(score[2]), 4),
                'sentiment_confidence': round(float(np.max(score)), 4)
            })

        # Progress logging every 10 batches
        if (i // batch_size) % 10 == 0:
            print(f"  Batch {i // batch_size + 1}/{total_batches} — {i + len(batch_texts)} reviews processed")

    return results


In [12]:
#  3.2. PREPROCESSING

def preprocess_reviews(df, text_col='reviews'):
    """Clean reviews before passing to model."""
    df = df.copy()
    df[text_col] = df[text_col].astype(str).str.strip()

    # Drop empty reviews
    df = df[df[text_col].str.len() > 0]
    df = df[df[text_col] != 'nan']

    return df.reset_index(drop=True)

### 4) Transformer Analysis

In [13]:

def run_transformer_analysis(df, borough_name, text_col='reviews', batch_size=32):
    """
    Full pipeline: preprocess → inference → attach results to df.

    Args:
        df          : Borough sample DataFrame
        borough_name: For logging
        text_col    : Column containing review text
        batch_size  : 32 for GPU, 8 for local CPU testing

    Returns:
        DataFrame with sentiment columns added
    """

    print(f"\n{'='*50}")
    print(f"Running transformer analysis: {borough_name}")
    print(f"{'='*50}")

    df = preprocess_reviews(df, text_col)
    print(f"Reviews after preprocessing: {len(df):,}")

    texts = df[text_col].tolist()

    start = time.time()
    sentiment_results = get_sentiment_batch(texts, batch_size=batch_size)
    elapsed = time.time() - start

    print(f"\nInference complete in {elapsed:.1f}s ({elapsed/len(texts)*1000:.1f}ms per review)")

    # Attach results
    results_df = pd.DataFrame(sentiment_results)
    df = pd.concat([df.reset_index(drop=True), results_df], axis=1)

    # Summary
    print(f"\nSentiment distribution:")
    print(df['sentiment_label'].value_counts(normalize=True).round(3))

    return df

In [14]:
manhattan_sample_df.head()

,listing_id,review_id,date,reviews,neighbourhood_group,room_type,price,review_length
0,6990,49986201,2015-10-08,"Cynthia is a very nice host , hospitable and s...",Manhattan,Private room,70.0,59
1,6990,19236,2009-12-05,What can I say about Cynthia?\r<br/>We stayed ...,Manhattan,Private room,70.0,143
2,6990,56559971,2015-12-14,Cynthia is: \r<br/>HONEST\r<br/>KIND\r<br/>HIG...,Manhattan,Private room,70.0,11
3,6990,538492,2011-09-17,In one word - WOW! I really had the time of my...,Manhattan,Private room,70.0,92
4,6990,12175934,2014-04-25,"Being a first time Airbnb user, I had a hard t...",Manhattan,Private room,70.0,186


In [15]:
manhattan_sample_df.columns

Index(['listing_id', 'review_id', 'date', 'reviews', 'neighbourhood_group',
       'room_type', 'price', 'review_length'],
      dtype='object')

### 5) Run Analysis

#### Test Locally First:

In [ ]:

#6. LOCAL TEST RUN (small sample, can be run on CPU - for logic verification)


# Test on 100 rows locally before full Colab run
BATCH_SIZE = 8   # small for CPU — change to 32 on Colab GPU

manhattan_test = manhattan_sample_df.head(100).copy()
manhattan_test_results = run_transformer_analysis(
    manhattan_test,
    borough_name='Manhattan (test)',
    batch_size=BATCH_SIZE
)

print("\nSample output:")
print(manhattan_test_results[['listing_id', 'reviews', 'sentiment_label',
                               'score_positive', 'score_neutral',
                               'score_negative', 'sentiment_confidence']].head(5))


Running transformer analysis: Manhattan (test)
Reviews after preprocessing: 100
  Batch 1/13 — 8 reviews processed
  Batch 11/13 — 88 reviews processed

Inference complete in 37.0s (369.7ms per review)

Sentiment distribution:
sentiment_label
Positive    0.87
Neutral     0.11
Negative    0.02
Name: proportion, dtype: float64

Sample output:
   listing_id                                            reviews  \
0        6990  Cynthia is a very nice host , hospitable and s...   
1        6990  What can I say about Cynthia?\r<br/>We stayed ...   
2        6990  Cynthia is: \r<br/>HONEST\r<br/>KIND\r<br/>HIG...   
3        6990  In one word - WOW! I really had the time of my...   
4        6990  Being a first time Airbnb user, I had a hard t...   

  sentiment_label  score_positive  score_neutral  score_negative  \
0        Positive          0.9820         0.0136          0.0045   
1        Positive          0.9775         0.0184          0.0041   
2         Neutral          0.4533         0

In [ ]:
manhattan_test_results.head()

,listing_id,review_id,date,reviews,neighbourhood_group,room_type,price,review_length,sentiment_label,score_negative,score_neutral,score_positive,sentiment_confidence
0,6990,49986201,2015-10-08,"Cynthia is a very nice host , hospitable and s...",Manhattan,Private room,70.0,59,Positive,0.0045,0.0136,0.9820,0.9820
1,6990,19236,2009-12-05,What can I say about Cynthia?\r<br/>We stayed ...,Manhattan,Private room,70.0,143,Positive,0.0041,0.0184,0.9775,0.9775
2,6990,56559971,2015-12-14,Cynthia is: \r<br/>HONEST\r<br/>KIND\r<br/>HIG...,Manhattan,Private room,70.0,11,Neutral,0.0747,0.4720,0.4533,0.4720
3,6990,538492,2011-09-17,In one word - WOW! I really had the time of my...,Manhattan,Private room,70.0,92,Positive,0.0032,0.0094,0.9874,0.9874
4,6990,12175934,2014-04-25,"Being a first time Airbnb user, I had a hard t...",Manhattan,Private room,70.0,186,Positive,0.0112,0.0774,0.9114,0.9114


#### Run analysis on complete dataset

In [16]:

# Run on entire dataset

# Change batch size and run on full samples
BATCH_SIZE = 32  # GPU

manhattan_transformer = run_transformer_analysis(manhattan_sample_df, 'Manhattan', batch_size=BATCH_SIZE)
brooklyn_transformer  = run_transformer_analysis(brooklyn_sample_df,  'Brooklyn',  batch_size=BATCH_SIZE)


Running transformer analysis: Manhattan
Reviews after preprocessing: 14,946
  Batch 1/468 — 32 reviews processed
  Batch 11/468 — 352 reviews processed
  Batch 21/468 — 672 reviews processed
  Batch 31/468 — 992 reviews processed
  Batch 41/468 — 1312 reviews processed
  Batch 51/468 — 1632 reviews processed
  Batch 61/468 — 1952 reviews processed
  Batch 71/468 — 2272 reviews processed
  Batch 81/468 — 2592 reviews processed
  Batch 91/468 — 2912 reviews processed
  Batch 101/468 — 3232 reviews processed
  Batch 111/468 — 3552 reviews processed
  Batch 121/468 — 3872 reviews processed
  Batch 131/468 — 4192 reviews processed
  Batch 141/468 — 4512 reviews processed
  Batch 151/468 — 4832 reviews processed
  Batch 161/468 — 5152 reviews processed
  Batch 171/468 — 5472 reviews processed
  Batch 181/468 — 5792 reviews processed
  Batch 191/468 — 6112 reviews processed
  Batch 201/468 — 6432 reviews processed
  Batch 211/468 — 6752 reviews processed
  Batch 221/468 — 7072 reviews proces

In [28]:
manhattan_transformer.head()

,listing_id,review_id,date,reviews,neighbourhood_group,room_type,price,review_length,sentiment_label,score_negative,score_neutral,score_positive,sentiment_confidence
0,6990,49986201,2015-10-08,"Cynthia is a very nice host , hospitable and s...",Manhattan,Private room,70.0,59,Positive,0.0045,0.0136,0.9820,0.9820
1,6990,19236,2009-12-05,What can I say about Cynthia?\r<br/>We stayed ...,Manhattan,Private room,70.0,143,Positive,0.0041,0.0184,0.9775,0.9775
2,6990,56559971,2015-12-14,Cynthia is: \r<br/>HONEST\r<br/>KIND\r<br/>HIG...,Manhattan,Private room,70.0,11,Neutral,0.0747,0.4720,0.4533,0.4720
3,6990,538492,2011-09-17,In one word - WOW! I really had the time of my...,Manhattan,Private room,70.0,92,Positive,0.0032,0.0094,0.9874,0.9874
4,6990,12175934,2014-04-25,"Being a first time Airbnb user, I had a hard t...",Manhattan,Private room,70.0,186,Positive,0.0112,0.0774,0.9114,0.9114


In [29]:
from pathlib import Path
import pandas as pd

# this finds the project root regardless of where the notebook is
ROOT = Path('/content/airbnb-reviews-nlp')
RESULTS_DIR = ROOT / "data" / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

manhattan_transformer.to_csv(RESULTS_DIR / "manhattan_transformer.csv", index=False)
brooklyn_transformer.to_csv(RESULTS_DIR / "brooklyn_transformer.csv", index=False)

Configure your Git and push results to Github

In [ ]:
# Step 1 — Configure git
!git config user.email "your@email.com"
!git config user.name "yourname"

In [ ]:
# Step 3 — Stage the results - Currently only containing transformer results
# Use !git status to check what changes are staged.
!git add data/results/

# Step 4 — Commit with message
!git commit -m "Add transformer sentiment analysis results for Manhattan and Brooklyn"

# Step 5 — Push new branch to GitHub
!git push --set-upstream origin your-branch-name